In [ ]:
# Make the control data from real XRT observations which are independent from the target observations
# The files are downloaded via Sciserver and moved to grappa
# The Files are contained in "/home/yu/XRT/MultipleTiling/swiftfiles/eventfiles_{year}" 

In [ ]:
# GTI 200 s cut
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.integrate import quad
from astropy.table import Table
import random
from tqdm import tqdm
from scipy.integrate import dblquad
import glob
import os
from pathlib import Path

years = np.array([2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]).astype(int)
for year in years:
    evt_dir = f"/home/yu/XRT/MultipleTiling/swiftfiles/eventfiles_{year}"
    evt_files = sorted(glob.glob(os.path.join(evt_dir, f"*pcw3*.evt")))
    for path in evt_files:
        p = Path(path)
        base = p.name.replace('xpcw3po_cl.evt', '')
        hdul = fits.open(path)
        event_data = hdul['EVENTS'].data
        gti = hdul['GTI'].data
        n = len(gti)
        for i in range(n):
            start = gti['START'][i]
            stop = gti['STOP'][i]
            exp = stop-start
            k=0
            while exp>=200:
                cut_start = start
                cut_stop = cut_start+200
                modified_data = event_data[(event_data['TIME']>cut_start) & (event_data['TIME']<=cut_stop)]
                # GTI creation
                newgti_start=np.array([cut_start]).astype('>f8')
                newgti_stop=np.array([cut_stop]).astype('>f8')

                event_table = Table(
                    [newgti_start, newgti_stop],
                    names=('START', 'STOP'),
                )

                bintablehdu = fits.BinTableHDU(data=event_table, name = 'GTI')
                created_gti = bintablehdu.data

                combined = fits.FITS_rec(np.concatenate([
                    created_gti.astype(gti.dtype)  # エンディアン揃えた上で
                ]))
                sort_idx = np.argsort(combined['START'])
                modified_gti=combined[sort_idx].astype(gti.dtype) 

                # 3. create new BinTableHDU with the modified data and the original column definition
                new_events_hdu = hdul['EVENTS'].copy()
                new_events_hdu.data = modified_data
                new_gti_hdu = hdul['GTI'].copy()
                new_gti_hdu.data = modified_gti
                # new_events_hdu = fits.BinTableHDU(data=modified_data, header=hdul['EVENTS'].header)
                # new_gti_hdu = fits.BinTableHDU(data=modified_gti, header=hdul['GTI'].header)
                # 4. reconstruct HDUList
                outdir = Path(f"/home/yu/XRT/MultipleTiling/swiftfiles/GTIcut_{year}")
                outdir.mkdir(parents=True, exist_ok=True)

                new_hdul = fits.HDUList([hdul[0], new_events_hdu, new_gti_hdu, hdul['BADPIX'], hdul['BIASDIFF']])
                new_hdul.writeto(f'{outdir}/{base}_GTI{i+1}_{k+1}.evt', overwrite=True)
                

                start = cut_stop
                exp=stop-start
                k=k+1


In [ ]:
# How many tiling observations exist within a threshold
# 'exist_evt_8.6e+04_200.npy' : real follow up tiling observations which have more than 200 s exposure within 1 day
# ex) IC15** has originally 7 tilings → only 3 tilings pass the threshold above　→ only the 3 files are written in 'exist_evt_8.6e+04_200.npy'

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.io import fits
import pandas as pd

full_csv = pd.read_csv('/home/yu/XRT/MultipleTiling//full_with_icname_swifttime_obsstart.csv')

base_dir = Path('/home/yu/XRT/MultipleTiling')
thr = [10**3, 10**4, 10**4.5, 86400, 10**5]
# thr = [86400]
timethr=200
for thr in thr:
    timelist = []
    
    enoughgti=[]
    
    # ICname → ObsStart 
    icname_to_obs_thr = {}
    icname_to_obsstart = {}
    icname_to_obsnum = {}
    for icname in full_csv["ICname"]:
        ToO = full_csv['SwiftMET'][full_csv['ICname']==icname].iloc[0]
        alert_dir = base_dir / icname
        evt_files = sorted(alert_dir.glob("**/*001xpcw3po_cl.evt"))
        print(icname, len(evt_files))
        enoughgtifile=[]
        ObsStarts=[]
        lackgti=[]
        for evt in evt_files:
            path = Path(evt)
            hdul = fits.open(path)
            gti = hdul['GTI'].data
            time = 0
            ObsStarts.append(gti['START'][0])
            for a,b in gti:
                if a >= ToO+thr:
                    break

                if b <= ToO+thr:
                    time+=(b-a)

                if (a < ToO+thr) & (b > ToO+thr):
                    time+=ToO+thr-a

            timelist.append(time)
            if time>=timethr:
                enoughgtifile.append(evt)
                enoughgti.append(time)
            else:
                lackgti.append(evt)
        dire = Path(f'/home/yu/XRT/MultipleTiling/{icname}')
        # dire.mkdir(exist_ok=True)
        np.save(f'{dire}/lack_evt_{thr:.1e}_{timethr}.npy', lackgti)
        np.save(f'{dire}/exist_evt_{thr:.1e}_{timethr}.npy', enoughgtifile)
    #     icname_to_obs_thr[icname] = len(enoughgtifile)
    #     icname_to_obsstart[icname] = float(np.min(ObsStarts))
    #     icname_to_obsnum[icname] = float(len(evt_files))
    # full_csv["ObsStart"] = full_csv["ICname"].map(icname_to_obsstart)
    # full_csv["ObsNum"] = full_csv["ICname"].map(icname_to_obsnum)
    # full_csv['WhenStartObs'] = full_csv['ObsStart']-full_csv['SwiftMET']
    # full_csv[f"Num{thr:.1e}_{timethr}"] = full_csv["ICname"].map(icname_to_obs_thr)
    # # 必要なら保存
    # full_csv.to_csv('/home/yu/XRT/MultipleTiling/full_with_icname_swifttime_obsstart.csv',
    #                 index=False)
            # enoughgtifile.append(evt)
    # plt.title(f'time of the GTIs (<{thr:.1e} s)', fontsize=20)
    # plt.xlabel('exposure time (s)', fontsize=17)
    # plt.ylabel('Nentry', fontsize=17)
    # plt.tick_params(labelsize=14)
    # plt.hist(timelist, bins = np.linspace(np.min(timelist), np.max(timelist), 50), label=f'all: {len(timelist)}')
    # plt.hist(enoughgti, bins = np.linspace(np.min(timelist), np.max(timelist), 50), histtype='step', linewidth=1.5,  label=f'> {timethr}s: {len(enoughgti)}')
    # # plt.hist(timelist, bins =np.logspace(np.log10(min(timelist)), np.log10(max(timelist)), 50), label=f'all: {len(timelist)}')
    # # plt.hist(enoughgti, bins =np.logspace(np.log2(min(timelist)), np.log10(max(timelist)), 2), histtype='step', linewidth=1.5,  label=f'> 200s: {len(enoughgti)}')
    # plt.legend(fontsize=17)
    # plt.tight_layout()
    # # plt.xscale('log')
    # plt.yscale('log')
    # plt.show()



In [ ]:
# This code assigns 1000 GTI cut files to each file randomly as control samples
# If the target observation is conducted in 2020, the control samples are chosen from 2019, 2020, and 2021

# IC**/combined/combined_i.npy includes paths of the control samples (If the number of tiles that pass the threshold (200s, 1day) is 3, combined_i.npy contains 3 paths that are randomly selected)
# make combine_1 to combine_1000 →statistics: 1000

from pathlib import Path
import numpy as np
import random
from tqdm import tqdm
import re
import shutil

base = Path('/home/yu/XRT/MultipleTiling')
exist_npy_name = 'exist_evt_8.6e+04_200.npy'

# directory of GTIcut files
gti_base = base / 'swiftfiles'   # 例: /home/yu/XRT/MultipleTiling/swiftfiles



SEED = 42

# ----------------

def ic_year_from_name(ic_name: str) -> int:
    """
    IC160731A -> 2016
    """
    # "IC" + YY + ...
    yy = int(ic_name[2:4])
    return 2000 + yy

def extract_target_id_from_exist_path(exist_path: str) -> str:
    """
    exist ファイル名の [5:10] を target として使う。
    """
    existname = Path(exist_path).name
    return str(existname)[5:10]

def collect_gti_files_for_years(gti_base: Path, years: list[int]) -> list[Path]:
    files = []
    for y in years:
        d = gti_base / f'GTIcut_{y}'
        if d.is_dir():

    return files

def copy_random_files_to_target(files: list[Path], new_base: Path, n_add: int, i):
    if not new_base.exists():
        new_base.mkdir(parents=True, exist_ok=True)

    if len(files) == 0:
        return 0

    k = min(n_add, len(files))
    chosen = random.sample(files, k)
    np.save(f'{new_base}/combined_{i}.npy', chosen)
    copied = 0
    bg_dir = new_base/ "BG"
    bg_dir.mkdir(parents=True, exist_ok=True)
    for src in chosen:
        dst = bg_dir / src.name

        shutil.copy2(src, dst)
        copied += 1
    return len(chosen)


def main():
    if SEED is not None:
        random.seed(SEED)

    ic_dirs = sorted([p for p in base.glob('IC*') if p.is_dir()])

    for icdir in tqdm(ic_dirs):
        ic_name = icdir.name
        npy_path = icdir / exist_npy_name
        if not npy_path.exists():
            print(f"[skip] npy not found: {npy_path}")
            continue

        existlist = np.load(npy_path, allow_pickle=True)

        N_ADD = len(existlist)
        # target id を抽出（重複除去）
        # targets = sorted({extract_target_id_from_exist_path(e) for e in existlist})

        year = ic_year_from_name(ic_name)
        years = [year - 1, year, year + 1]

        gti_files = collect_gti_files_for_years(gti_base, years)
        # print(f"\n{ic_name}: years={years}, GTI files={len(gti_files)}, targets={len(targets)}")

        if len(gti_files) == 0:
            print("  [warn] No GTIcut files found for these years.")
            continue

        for i in range(1000):
            new_base = icdir / 'combined' /f'combined_{i}'
            # new_base.mkdir(exist_ok=True)

            n_copied = copy_random_files_to_target(gti_files, new_base, N_ADD, i)
            

if __name__ == "__main__":
    main()


100%|██████████| 1/1 [00:03<00:00,  3.21s/it]


In [ ]:
#ex)
a=np.load('/home/yu/XRT/MultipleTiling/IC251210A/combined/combined_1/combined_1.npy', allow_pickle=True)
print((a))

# We just copy this passes in combined_*.npy file to get BG control samples corresponding to 1 set of tiling observbations for an alert
# The BG files are contained in '/home/yu/XRT/MultipleTiling/IC**/combined/combined_i/BG/


[PosixPath('/home/yu/XRT/MultipleTiling/swiftfiles/GTIcut_2024/sw00049844003_GTI1_4.evt')
 PosixPath('/home/yu/XRT/MultipleTiling/swiftfiles/GTIcut_2024/sw00049502072_GTI1_6.evt')
 PosixPath('/home/yu/XRT/MultipleTiling/swiftfiles/GTIcut_2024/sw00049088002_GTI19_1.evt')
 PosixPath('/home/yu/XRT/MultipleTiling/swiftfiles/GTIcut_2024/sw00046250005_GTI1_6.evt')]
